# Daily Challenge: LangChain Pipelines with Open-Source LLMs (Student)
Use this guided notebook with TODOs. Runs on CPU with small HF models (e.g., flan-t5-small).

## What you'll learn
- Set up LangChain with lightweight open-source models.
- Build an LLMChain using a prompt template.
- Compose a two-step Runnable pipeline (summary ? bullets).
- Bonus: add a simple conversation chain with memory.

## What you will create
- Installed environment for LangChain + transformers.
- LLMChain that rewrites text in a simpler style.
- Runnable pipeline that summarizes then bullet-izes text.
- (Bonus) Conversation chain showing memory.

## Part 1: Environment setup (fast)
Install needed packages. CPU is fine for tiny models.

In [1]:
import torch
# Verify if a GPU is available, otherwise default to CPU
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Using device: cpu


In [11]:
# Install necessary libraries for LangChain and Transformers
!pip install langchain==0.1.7 langchain-community==0.0.20 langchain-core==0.1.23 transformers==4.37.2 accelerate

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.4/129.4 kB 5.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 815.9/815.9 kB 32.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 68.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 241.2/241.2 kB 15.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 100.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 35.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.4/55.4 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 85.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.0/53.0 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 98.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 4.1 MB/s eta 0:00:00
  Attempting uninstall: tenacity
    Found existing

In [ ]:
# Restart the runtime to ensure new packages are loaded
import os
os._exit(0)

## Part 2: Load a tiny model and build your first LLMChain
Use a small model (e.g., google/flan-t5-small) to keep inference quick.

In [1]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline
from langchain_community.llms import HuggingFacePipeline
from langchain_core.prompts import PromptTemplate
from langchain.chains import LLMChain

print("Libraries imported successfully.")

Libraries imported successfully.


In [ ]:

# TODO: choose a small model
model_name = "google/flan-t5-small"  # keep small for CPU


In [ ]:

# TODO: load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)


In [4]:
# Ensure model and tokenizer are defined before creating the pipeline
model_name = "google/flan-t5-small"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

gen_pipeline = pipeline(
    task="text2text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=128,
)
llm = HuggingFacePipeline(pipeline=gen_pipeline)
print("LLM pipeline and model initialized successfully.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/308M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

LLM pipeline and model initialized successfully.


In [ ]:

# TODO: build prompt + LLMChain for friendly rewriting
template = "Rewrite this text to be simpler for beginners:{text}"
prompt = PromptTemplate(template=template, input_variables=["text"])
chain = LLMChain(prompt=prompt, llm=llm)

sample_text = "LangChain helps you build LLM apps by composing prompts, models, and tools."
rewritten = chain.run(text=sample_text)
print(rewritten)


## Part 3: Two-step pipeline (summary ? bullets)
Summarize a paragraph, then turn it into 3 bullets using the same LLM.

In [9]:
from langchain_core.runnables import RunnableLambda
from langchain_core.prompts import PromptTemplate

# Step 1 Template: Summarize the provided text into a single short sentence
summary_prompt = PromptTemplate(
    template="Summarize the following text into one short sentence:\n\n{paragraph}",
    input_variables=["paragraph"],
)

# Step 2 Template: Take the summary and format it as 3 bullet points
bullets_prompt = PromptTemplate(
    template="Take the following summary and turn it into 3 short bullet points:\n\n{summary}",
    input_variables=["summary"],
)

In [5]:
# Ensure prompts are defined before building the chain
summary_prompt = PromptTemplate(
    template="Summarize the following text into one short sentence:\n\n{paragraph}",
    input_variables=["paragraph"],
)

bullets_prompt = PromptTemplate(
    template="Take the following summary and turn it into 3 short bullet points:\n\n{summary}",
    input_variables=["summary"],
)

# Build the LCEL pipeline
summary_chain = summary_prompt | llm

summarize_then_bullets = (
    {"summary": summary_chain}
    | bullets_prompt
    | llm
)
print("LCEL pipeline defined successfully.")

LCEL pipeline defined successfully.


In [7]:
# Re-defining variables that might have been lost in the restart
paragraph = """LangChain is a framework for building applications with large language models by composing prompts, models, and tools. It supports chains, agents, and retrieval workflows."""

# Execute the pipeline
# Note: ensure cell 4c718436 was run to define summarize_then_bullets
try:
    bullets_output = summarize_then_bullets.invoke({"paragraph": paragraph})
    print("Final Output:")
    print(bullets_output)
except NameError:
    print("Error: Please ensure the cell defining 'summarize_then_bullets' (cell 4c718436) has been executed.")

Final Output:
LangChain is a framework for building applications with large language models by composing prompts, models, and tools.


## Part 4 (Bonus): Conversation chain with memory
Show how two turns keep context.

In [6]:
from langchain.chains import ConversationChain
from langchain.memory import ConversationBufferMemory

# Initialize memory
memory = ConversationBufferMemory()

# Create conversation chain
convo = ConversationChain(llm=llm, memory=memory, verbose=False)

# Execute turns
reply1 = convo.predict(input="Hi there! What is LangChain?")
print("Turn 1:", reply1)

reply2 = convo.predict(input="Can it help me build a simple chatbot?")
print("Turn 2:", reply2)

Turn 1: LangChain LangChain is a village in the administrative district of Gmina Gmina Gmina Gmina Gmina Gmina Gmina Gmina Gmina Gmina Gmina Gmina Gmina Gmina Gmina Gmina Gmina Gmina Gmina Gmina Gmina Gmina Gmina Gmina Gmina Gmina Gmina Gmina Gmina Gmina Gmina Gmina Gmina Gmina Gmina Gmina Gmina
Turn 2: It's a simple chatbot that can be used to create a chatbot.


## Your observations
- **Latency**: Using `flan-t5-small` on CPU is relatively fast (typically < 2 seconds per generation) because the model has only ~60M parameters.
- **Quality**: The model is good at basic transformations but might struggle with complex reasoning or very long context due to its size.
- **Quirks**: It can sometimes be overly brief or produce repetitive bullet points if the summary is too short.